In [1]:
import polars as pl
import torch


players_df = pl.read_csv('csv/player.csv')

all_ids = players_df['id'].unique()
null_player_id = 0
nba_id_to_idx_mapping = {}
for idx, nba_id in enumerate(all_ids):
    nba_id_to_idx_mapping[nba_id] = idx + 1 # reserve 0 for null player id

nba_id_to_idx_mapping[null_player_id] = 0

max_nba_id = max(nba_id_to_idx_mapping.keys())
lookup_tensor = torch.zeros(max_nba_id + 1, dtype=torch.long)
for nba_id, sequential_idx in nba_id_to_idx_mapping.items():
    lookup_tensor[nba_id] = sequential_idx
    

In [11]:
from datasets import Dataset, load_dataset
from torch.utils.data import DataLoader

REPO_ID = 'sviridov/nba-posession-processed'
processed_dataset = load_dataset(REPO_ID, split='train')

processed_dataset.set_format("torch")

loader = DataLoader(
    processed_dataset, 
    batch_size=2048,      # Critical for GPU utilization
    shuffle=True, 
    num_workers=4,         
    pin_memory=True,       
    persistent_workers=True # Keeps data loading alive between epochs
)


print(len(loader) * 2048, " training shot cycles")

5349376  training shot cycles


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [ ]:
from NBADeepFm import NBATransformer

model = NBATransformer(num_players=len(nba_id_to_idx_mapping), embed_dim=32)
model = model.to(device)


In [ ]:
# train the model
from torch.amp import autocast, GradScaler
from torch import nn

scaler = GradScaler()
criterion = nn.CrossEntropyLoss().to(device)
identity_criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

num_players = len(nba_id_to_idx_mapping)
MASK_ID = num_players # Use the last index as the mask
MASK_ID = torch.tensor(MASK_ID, dtype=torch.long).to(device)

num_epochs = 1
batch_size = 64

for epoch in range(num_epochs):
    for i, batch in enumerate(loader):
        
        lineup = batch['lineup_ids'].to(device)
        roles = batch['role_ids'].to(device)
        targets = batch['labels'].to(device)
        

        mask_indices = (torch.rand(lineup.size(0), device=device) < 0.2) # Create directly on GPU

        input_roles = roles.clone()
        original_shooters = input_roles[mask_indices, 0].clone() # Save original shooter IDs
        input_roles[mask_indices, 0] = MASK_ID # Mask the shooter role

        optimizer.zero_grad()

        with autocast(device_type=device):
            point_logits, id_logits = model(lineup, input_roles)
            loss_points = criterion(point_logits, targets)
            loss_identity = identity_criterion(id_logits, original_shooters)
            loss = loss_points + 0.5 * loss_identity

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        if (i+1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(loader)}], Loss: {loss.item():.4f}')
    
    torch.save(model, f"nba_transformer_epoch{epoch}_checkpoint.pth")
            


print("Finished training model")

/tmp/ipykernel_140693/664922028.py:5: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = GradScaler()


NameError: name 'device' is not defined

In [ ]:
#upload to hugging face hub


# model.push_to_hub("sviridov/nba-transformer", private=True)